# GROTH-05b : croissance des algèbres et dimension de Gelfand–Kirillov

**Navigation** : [<< Série SymbolicAI/Lean](../README.md) · [< GROTH-05 Séries formelles multivariées](GROTH-05-MvPowerSeries-Lean.ipynb) · [> GROTH-06 Groupes formels natifs](GROTH-06-FormalGroups-Lean.ipynb)

Accrétion de [GROTH-05](GROTH-05-MvPowerSeries-Lean.ipynb) : la série lit les objets par leurs coefficients, ce carnet fait compter ces coefficients à des dimensions. Il calcule la fonction de Hilbert de quatre familles d'algèbres, en estime la dimension de Gelfand–Kirillov, énonce le théorème du trou (Smoktunowicz, 2006) et mesure la frontière de Mathlib. Grain du point 1 de [#27589](https://github.com/jsboige/CoursIA/issues/27589).

## Introduction : pourquoi un carnet sur la croissance des algèbres

En novembre 2026, une note de recherche de J. Stavrianos a rapporté la première formalisation connue d'un théorème d'Agata Smoktunowicz daté de 2006 : une preuve de l'ordre de deux mille lignes de Lean, produite en une heure et demie par un agent de preuve automatique, vérifiées en salle blanche (axiomes standard seulement, aucun `sorry`, aucun `native_decide`). La note est explicite sur un point qu'il ne faut pas perdre : c'est une **preuve formelle du théorème**, pas une formalisation de la preuve publiée ; les deux ne coïncident que sur les étapes d'ouverture.

Avant de lire une telle preuve, il faut savoir de quoi elle parle. Les objets sont des algèbres graduées et leur **croissance** : combien d'éléments indépendants peut-on fabriquer avec au plus $n$ multiplications ? La mesure est la **dimension de Gelfand–Kirillov**, une quantité calculable. Ce carnet la calcule là où elle est connue, énonce le théorème du trou, et marque honnêtement ce qu'il ne fait pas.

Pourquoi dans l'îlot GROTH : [GROTH-05](GROTH-05-MvPowerSeries-Lean.ipynb) décompose une série formelle par ses coefficients. La série de Hilbert d'une algèbre graduée, $\sum_n \dim(A_n)\, t^n$, a précisément pour coefficients les dimensions graduées que ce carnet compte — c'est le même substrat, lu par les coefficients.

## 1. La dimension de Gelfand–Kirillov : définition

Soit $A$ une algèbre de type fini sur un corps, engendrée par $x_1, \dots, x_d$. On note $A_n$ le sous-espace vectoriel engendré par les mots de longueur au plus $n$ en ces générateurs, et $a_n = \dim A_n$ la **fonction de Hilbert** cumulée.

La dimension de Gelfand–Kirillov est

$$\mathrm{GKdim}(A) = \limsup_{n \to \infty} \frac{\log a_n}{\log n}.$$

Trois régimes en découlent :

- si $a_n \sim c\, n^{\delta}$ (croissance polynomiale), alors $\mathrm{GKdim}(A) = \delta$ ;
- si la croissance est exponentielle, le rapport $\log a_n / \log n$ diverge et $\mathrm{GKdim}(A) = \infty$ ;
- pour une algèbre de type fini, la réciproque vaut : $\mathrm{GKdim}(A) < \infty$ si et seulement si la croissance est polynomiale.

Une convention utile pour les algèbres graduées par $\mathbb{N}$, avec $A_0$ de dimension finie : si la composante de degré $n$ seule, notée $h(n)$, croît comme $n^{\delta - 1}$, alors la cumulée croît comme $n^{\delta}$ et $\mathrm{GKdim}(A) = \delta$. C'est la normalisation des articles d'Artin–Stafford et de Smoktunowicz : « croissance quadratique » y désigne $h(n) \sim n^2$, donc une dimension GK de 3.

Le carnet mesure deux signatures de croissance : les **pentes** $\log a_n / \log n$, qui tendent vers $\delta$ en régime polynomial — mais logarithmiquement lentement, on le verra — et divergent en régime exponentiel ; et les **ratios** $a_{n+1}/a_n$, qui tendent vers 1 en régime polynomial et vers une constante $> 1$ en régime exponentiel. Le ratio est de loin le diagnostic le plus fiable aux petites valeurs de $n$.

In [1]:
from fractions import Fraction
from itertools import product
from math import comb, factorial, log


def pente_gk(dimensions):
    """Pentes locales log(dim A_n) / log(n), pour n = 1, 2, ..."""
    return [round(log(d) / log(n), 3)
            for n, d in enumerate(dimensions, start=1) if d > 1 and n > 1]


def ratios_successifs(dimensions):
    """Rapports dim(n+1)/dim(n) : tendent vers 1 si la croissance est polynomiale."""
    return [round(b / a, 3) for a, b in zip(dimensions, dimensions[1:])]


def afficher(nom, dimensions):
    print(f"{nom} : dim A_n = {dimensions}")
    print(f"  pentes  log(dim)/log(n) : {pente_gk(dimensions)}")
    print(f"  ratios  dim(n+1)/dim(n) : {ratios_successifs(dimensions)}")

## 2. La référence commutative : les anneaux de polynômes

Pour $k[x_1, \dots, x_d]$, les mots de longueur au plus $n$ sont les monômes de degré total au plus $n$, et leur nombre est le coefficient binomial $\binom{n+d}{d}$. La croissance est polynomiale de degré $d$, donc $\mathrm{GKdim} = d$ : les anneaux de polynômes réalisent **tous** les degrés entiers, et la dimension GK se comporte pour eux comme la dimension de Krull.

Le premier calcul vérifie la formule par énumération explicite des multi-exposants, sans jamais utiliser le binôme.

In [2]:
def monomes(d, n):
    """Nombre de monomes de degre total <= n en d variables (enumeration)."""
    return sum(1 for exps in product(range(n + 1), repeat=d) if sum(exps) <= n)


for d in (2, 3):
    calcule = [monomes(d, n) for n in range(1, 8)]
    formule = [comb(n + d, d) for n in range(1, 8)]
    verdict = "identique" if calcule == formule else "ECART"
    print(f"d = {d} : enumeration      = {calcule}")
    print(f"        formule C(n+{d},{d}) = {formule}  -> {verdict}")

d = 2 : enumeration      = [3, 6, 10, 15, 21, 28, 36]
        formule C(n+2,2) = [3, 6, 10, 15, 21, 28, 36]  -> identique
d = 3 : enumeration      = [4, 10, 20, 35, 56, 84, 120]
        formule C(n+3,3) = [4, 10, 20, 35, 56, 84, 120]  -> identique


Les deux listes coïncident terme à terme. Pour $d = 2$ : 3, 6, 10, 15, 21, 28, 36 ; pour $d = 3$ : 4, 10, 20, 35, 56, 84, 120. Ce sont bien $\binom{n+2}{2}$ et $\binom{n+3}{3}$.

L'énumération est volontairement naïve : elle compte des $d$-uplets d'exposants, pas des monômes. C'est le même geste que la lecture par coefficients de GROTH-05 — on énumère ce qui est petit, on laisse la formule fermer le compte.

In [3]:
for d in (2, 3):
    afficher(f"k[x_1..x_{d}]", [monomes(d, n) for n in range(1, 11)])

k[x_1..x_2] : dim A_n = [3, 6, 10, 15, 21, 28, 36, 45, 55, 66]
  pentes  log(dim)/log(n) : [2.585, 2.096, 1.953, 1.892, 1.86, 1.842, 1.831, 1.824, 1.82]
  ratios  dim(n+1)/dim(n) : [2.0, 1.667, 1.5, 1.4, 1.333, 1.286, 1.25, 1.222, 1.2]
k[x_1..x_3] : dim A_n = [4, 10, 20, 35, 56, 84, 120, 165, 220, 286]
  pentes  log(dim)/log(n) : [3.322, 2.727, 2.565, 2.501, 2.473, 2.46, 2.455, 2.455, 2.456]
  ratios  dim(n+1)/dim(n) : [2.5, 2.0, 1.75, 1.6, 1.5, 1.429, 1.375, 1.333, 1.3]


Premier constat, contre-intuitif : les pentes **ne valent pas** 2 et 3. À $n = 2$ elles sur-estiment (2.585 et 3.322 pour des cibles de 2 et 3), puis elles sous-estiment durablement : 1.82 et 2.456 à $n = 10$. Les ratios, eux, descendent régulièrement vers 1 — 2.0, 1.667, 1.5, ..., 1.2 pour $k[x,y]$ ; 2.5, 2.0, 1.75, ..., 1.3 pour $k[x,y,z]$. C'est la signature du régime polynomial : ajouter une couche de mots coûte de moins en moins, relativement.

La dimension GK est donc le degré de croissance, et non le nombre de générateurs : $k[x,y]$ a deux générateurs et dimension 2, mais $k[x,y,z]$ a trois générateurs et dimension 3 parce que les relations sont triviales. La suite montre que la réciproque est fausse.

La pente finie n'est donc pas une mesure : elle converge vers $\delta$ en $1/\log n$, ce qui est extrêmement lent. La cellule suivante sonde la formule fermée à des valeurs croissantes de $n$ pour rendre cette lenteur visible ; c'est le garde-fou contre l'erreur la plus répandue de la dimension GK — lire une pente à $n = 10$ et l'appeler une dimension.

In [4]:
print("sonde asymptotique -- pente estimee sur la formule fermee :")
for d in (2, 3):
    mesures = []
    for n in (10, 10 ** 2, 10 ** 4, 10 ** 6):
        dimension = comb(n + d, d)
        mesures.append(f"n = {n:>7} : {log(dimension) / log(n):.3f}")
    print(f"  d = {d} (cible {d}) -> " + " | ".join(mesures))

sonde asymptotique -- pente estimee sur la formule fermee :
  d = 2 (cible 2) -> n =      10 : 1.820 | n =     100 : 1.856 | n =   10000 : 1.925 | n = 1000000 : 1.950
  d = 3 (cible 3) -> n =      10 : 2.456 | n =     100 : 2.624 | n =   10000 : 2.806 | n = 1000000 : 2.870


Pour une cible de 2, la pente vaut 1.820 à $n = 10$, 1.856 à $n = 100$, 1.925 à $n = 10^4$ et seulement 1.950 à $n = 10^6$. Pour une cible de 3 : 2.456, 2.624, 2.806, puis 2.870. Même à un million, un tiers de la distance à la cible reste à combler ($3 - 2.870$).

La conséquence est nette, et elle vaut pour tout ce carnet : ce qui se **mesure**, c'est le nombre exact $\dim A_n$ — par énumération, par formule binomiale, ou par un rang calculé sur $\mathbb{Q}$. La dimension GK, elle, se **lit** sur la croissance asymptotique, jamais sur une pente à $n$ petit. Les sections suivantes gardent ce principe : le compte est mesuré, la dimension est conclue.

## 3. L'algèbre libre : croissance exponentielle, dimension infinie

Dans l'algèbre libre $k\langle x_1, \dots, x_d\rangle$, il n'y a aucune relation : les mots de longueur exactement $n$ sont au nombre de $d^n$, et la cumulée vaut

$$a_n = 1 + d + \dots + d^n = \frac{d^{n+1} - 1}{d - 1}.$$

C'est une croissance **exponentielle**. La dimension de Gelfand–Kirillov est donc infinie — et c'est un piège classique de croire que l'algèbre libre en deux générateurs serait l'exemple non commutatif de dimension 2. Elle est l'exemple de croissance exponentielle, et le théorème du trou ne la contredit pas : $\infty$ est bien « au moins 3 ».

In [5]:
def mots_libres(d, n):
    """Nombre de mots de longueur <= n sur d generateurs (algebre libre)."""
    return sum(d ** k for k in range(n + 1))


for d in (2, 3):
    dimensions = [mots_libres(d, n) for n in range(1, 11)]
    afficher(f"algebre libre a {d} generateurs", dimensions)
    formule = [(d ** (n + 1) - 1) // (d - 1) for n in range(1, 11)]
    print(f"  formule (d^(n+1)-1)/(d-1) : {formule}")

algebre libre a 2 generateurs : dim A_n = [3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047]
  pentes  log(dim)/log(n) : [2.807, 2.465, 2.477, 2.574, 2.704, 2.848, 2.999, 3.154, 3.311]
  ratios  dim(n+1)/dim(n) : [2.333, 2.143, 2.067, 2.032, 2.016, 2.008, 2.004, 2.002, 2.001]
  formule (d^(n+1)-1)/(d-1) : [3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047]
algebre libre a 3 generateurs : dim A_n = [4, 13, 40, 121, 364, 1093, 3280, 9841, 29524, 88573]
  pentes  log(dim)/log(n) : [3.7, 3.358, 3.459, 3.664, 3.905, 4.16, 4.422, 4.685, 4.947]
  ratios  dim(n+1)/dim(n) : [3.25, 3.077, 3.025, 3.008, 3.003, 3.001, 3.0, 3.0, 3.0]
  formule (d^(n+1)-1)/(d-1) : [4, 13, 40, 121, 364, 1093, 3280, 9841, 29524, 88573]


Le contraste avec la section précédente est net, et il faut le lire dans le bon ordre. Pour $d = 2$, les pentes descendent d'abord (2.807 à $n = 2$, 2.465 à $n = 3$) puis **remontent sans borne** : 2.477, 2.574, 2.704, 2.848, 2.999, 3.154, 3.311 à $n = 10$ — et elles continuent, comme $n \ln 2 / \ln n$. Pour $d = 3$, même profil à un cran plus haut : 3.7, 3.358, puis 4.947 à $n = 10$.

Les ratios, eux, tendent vers une constante strictement supérieure à 1 : 2.333, 2.143, ..., 2.001 pour $d = 2$ ; 3.25, 3.077, ..., 3.0 pour $d = 3$. C'est la signature du régime exponentiel, l'exact opposé du régime polynomial où ils tendent vers 1. Aux petites valeurs de $n$, c'est le ratio — et non la pente — qui tranche : une pente de 3.3 ressemble à une dimension finie, un ratio qui ne descend pas sous 2 ne trompe personne.

Deux conséquences pour la lecture du théorème à venir. D'abord, la dimension GK mesure la **croissance polynomiale** : une algèbre de croissance exponentielle a une dimension infinie, pas une grande dimension finie. Ensuite, les vrais exemples non commutatifs de dimension 2 sont plus fins que l'algèbre libre — il faut des relations qui coupent l'exponentielle sans aplatir entièrement. Les deux sections suivantes en donnent deux, calculés.

## 4. L'algèbre de Weyl $A_1$, par formes normales

Soit $A_1 = k\langle x, y\rangle / (xy - yx - 1)$. La relation permet de réécrire tout mot en forme normale $x^i y^j$ : de $yx = xy - 1$ on tire

$$y^j x^p = \sum_{k=0}^{\min(j,p)} (-1)^k\, k!\, \binom{j}{k}\binom{p}{k}\, x^{p-k} y^{\,j-k}.$$

Les monômes $x^i y^j$ de poids $i + j \le n$ sont au nombre de $\frac{(n+1)(n+2)}{2}$ : même compte que $k[x,y]$, donc même dimension GK, 2. L'algèbre de Weyl est intègre mais **non graduée** (la relation est inhomogène) : les hypothèses du théorème du trou ne sont pas réunies, et c'est pourquoi elle n'est pas un contre-exemple.

Le calcul implémente la multiplication sur des dictionnaires de coefficients entiers et vérifie la relation à la machine, plutôt que de la postuler.

In [6]:
def weyl_mul(p, q):
    """Produit dans A_1 = k<x,y>/(xy - yx - 1).
    Un element est un dict {(i, j): coefficient entier}, forme normale x^i y^j."""
    r = {}
    for (i, j), a in p.items():
        for (p_, q_), b in q.items():
            for k in range(min(j, p_) + 1):
                coef = (-1) ** k * factorial(k) * comb(j, k) * comb(p_, k)
                cle = (i + p_ - k, j - k + q_)
                r[cle] = r.get(cle, 0) + a * b * coef
    return {cle: v for cle, v in r.items() if v}


X = {(1, 0): 1}
Y = {(0, 1): 1}

xy = weyl_mul(X, Y)
yx = weyl_mul(Y, X)
difference = {cle: xy.get(cle, 0) - yx.get(cle, 0) for cle in set(xy) | set(yx)}
print("xy - yx =", {cle: v for cle, v in difference.items() if v})
print("y^2 x^2 =", weyl_mul({(0, 2): 1}, {(2, 0): 1}))

xy - yx = {(0, 0): 1}
y^2 x^2 = {(2, 2): 1, (1, 1): -4, (0, 0): 2}


La relation tombe juste : `xy - yx = {(0, 0): 1}`, c'est-à-dire la constante 1, l'unité de l'algèbre. Le contrôle suivant, $y^2 x^2$, développe la formule à la main : $x^2 y^2 - 4xy + 2$, soit les trois monômes attendus avec leurs coefficients.

La machine confirme l'algèbre ; elle ne remplace pas la preuve que ces formes normales forment une base. C'est un fait de théorie (la forme normale de l'algèbre de Weyl), et le calcul de rang qui suit le vérifie seulement jusqu'à un degré borné.

In [7]:
def puissance_weyl(element, n):
    # Puissance n-ieme dans A_1 : n produits par weyl_mul, chacun remis sous forme
    # normale (x^i y^j). Aucune formule n'est posee : le resultat se calcule.
    resultat = {(0, 0): 1}
    for _ in range(n):
        resultat = weyl_mul(resultat, element)
    return resultat


x_plus_y = {(1, 0): 1, (0, 1): 1}
for n in range(1, 7):
    developpe = puissance_weyl(x_plus_y, n)
    poids_max = max(i + j for i, j in developpe)  # plus grand poids total atteint
    # Derniere colonne : nombre de monomes de la base x^i y^j de poids total <= n,
    # c'est-a-dire la taille maximale que peut avoir le support de (x+y)^n.
    print(f"(x+y)^{n} : {len(developpe)} monomes, poids maximal {poids_max}, "
          f"monomes de poids <= n : {(n + 1) * (n + 2) // 2}")

(x+y)^1 : 2 monomes, poids maximal 1, monomes de poids <= n : 3
(x+y)^2 : 4 monomes, poids maximal 2, monomes de poids <= n : 6
(x+y)^3 : 6 monomes, poids maximal 3, monomes de poids <= n : 10
(x+y)^4 : 9 monomes, poids maximal 4, monomes de poids <= n : 15
(x+y)^5 : 12 monomes, poids maximal 5, monomes de poids <= n : 21
(x+y)^6 : 16 monomes, poids maximal 6, monomes de poids <= n : 28


Le développement de $(x+y)^n$ retombe sur des poids $n, n-2, \dots$ : 2, 4, 6, 9, 12 et 16 monômes pour $n = 1..6$. Les monômes de poids au plus $n$ sont bien en nombre $\frac{(n+1)(n+2)}{2}$ — 3, 6, 10, 15, 21, 28 — la taille de la base candidate.

Reste à mesurer la dimension réelle de l'espace engendré : on réduit **tous** les mots de longueur au plus $n$, on les écrit dans la base des $x^i y^j$ de poids au plus $n$, et on calcule le rang sur $\mathbb{Q}$ par élimination de Gauss. Si le rang égale $\frac{(n+1)(n+2)}{2}$, la base candidate est indépendante jusqu'à ce degré.

In [8]:
def rang_sur_q(matrice):
    """Rang d'une matrice a coefficients rationnels (elimination de Gauss)."""
    m = [[Fraction(x) for x in ligne] for ligne in matrice]  # sur Q : aucun arrondi
    lignes = len(m)
    cols = len(m[0]) if m else 0
    rang = 0
    for c in range(cols):
        # 1. chercher un pivot non nul dans la colonne c, a partir de la ligne `rang`
        pivot = next((r for r in range(rang, lignes) if m[r][c] != 0), None)
        if pivot is None:
            continue  # colonne combinee des precedentes : elle n'augmente pas le rang
        # 2. l'amener sur la ligne `rang` et le ramener a 1
        m[rang], m[pivot] = m[pivot], m[rang]
        v = m[rang][c]
        m[rang] = [x / v for x in m[rang]]
        # 3. annuler la colonne c sur toutes les autres lignes
        for r in range(lignes):
            if r != rang and m[r][c] != 0:
                f = m[r][c]
                m[r] = [x - f * y for x, y in zip(m[r], m[rang])]
        rang += 1
    return rang


def mots_jusqua(n, alphabet="xy"):
    """Tous les mots de longueur <= n sur l'alphabet donne."""
    # Engendres par longueurs croissantes : la frontiere de longueur k + 1 s'obtient
    # en allongeant d'une lettre chaque mot de la frontiere de longueur k.
    yield ""
    frontiere = [""]
    for _ in range(n):
        frontiere = [mot + lettre for mot in frontiere for lettre in alphabet]
        yield from frontiere


def mot_vers_weyl(mot):
    # Remplacer chaque lettre par X ou Y, puis multiplier dans A_1 :
    # l'element obtenu se lit comme combinaison des monomes x^i y^j.
    element = {(0, 0): 1}
    for lettre in mot:
        element = weyl_mul(element, X if lettre == "x" else Y)
    return element


for n in range(1, 7):
    # Base de l'espace des elements de poids total <= n, puis une ligne de matrice
    # par mot de longueur <= n, lue dans cette base. Le rang de la matrice est la
    # dimension de l'image des mots de longueur <= n dans A_1.
    base = [(i, j) for i in range(n + 1) for j in range(n + 1) if i + j <= n]
    index = {monome: k for k, monome in enumerate(base)}
    matrice = []
    for mot in mots_jusqua(n):
        element = mot_vers_weyl(mot)
        ligne = [0] * len(base)
        for (i, j), v in element.items():
            ligne[index[(i, j)]] += v
        matrice.append(ligne)
    mesure = rang_sur_q(matrice)
    formule = (n + 1) * (n + 2) // 2  # ce que predit la forme normale x^i y^j
    print(f"n = {n} : {len(matrice)} mots reduits, rang = {mesure}, "
          f"formule = {formule} -> {'identique' if mesure == formule else 'ECART'}")

n = 1 : 3 mots reduits, rang = 3, formule = 3 -> identique
n = 2 : 7 mots reduits, rang = 6, formule = 6 -> identique
n = 3 : 15 mots reduits, rang = 10, formule = 10 -> identique
n = 4 : 31 mots reduits, rang = 15, formule = 15 -> identique
n = 5 : 63 mots reduits, rang = 21, formule = 21 -> identique
n = 6 : 127 mots reduits, rang = 28, formule = 28 -> identique


Le rang égale la formule pour $n = 1$ à $6$ : 3, 6, 10, 15, 21, 28. Jusqu'à ce degré, les mots de longueur au plus $n$ de $A_1$ engendrent exactement $\frac{(n+1)(n+2)}{2}$ dimensions — la dimension de $A_n$ est **calculée**, pas seulement annoncée.

Ce que le calcul prouve, et ce qu'il ne prouve pas : il prouve l'égalité pour ces six valeurs, par une élimination exacte sur $\mathbb{Q}$. Il ne prouve pas la formule pour tout $n$ — c'est le rôle de l'argument de forme normale, hors de portée d'un carnet de calcul. La frontière est franche, et c'est celle que tout le carnet tient.

## 5. Le plan quantique $k_q[x,y]$

Soit $k_q[x,y] = k\langle x, y\rangle / (xy - q\, yx)$ avec $q \neq 0$. La relation est **homogène** de degré 2 : l'algèbre est graduée par $\mathbb{N}$, sa composante de degré zéro est $k$ (de dimension 1), et c'est un domaine. Les hypothèses du théorème du trou sont donc réunies.

La forme normale est encore $x^i y^j$ : en faisant passer $y$ devant $p$ lettres $x$, on multiplie par $q^{-p}$ à chaque lettre traversée, donc $y^j x^p = q^{-jp}\, x^p y^j$. Le compte des monômes de poids au plus $n$ est $\frac{(n+1)(n+2)}{2}$, inchangé : la dimension GK vaut 2, exactement sur la **frontière** de la branche « au plus 2 » du théorème. C'est un témoin de saturation.

In [9]:
def plan_quantique_mul(q):
    """Produit dans k_q[x,y] = k<x,y>/(xy - q yx), q non nul.
    Forme normale x^i y^j, avec y^j x^p = q^(-j*p) x^p y^j."""
    def mul(p, r):
        # Produit terme a terme : (x^i y^j)(x^p y^q_) = q^(-j*p) x^(i+p) y^(j+q_).
        # Le facteur q^(-j*p) est le prix du passage des y^j a gauche des x^p.
        sortie = {}
        for (i, j), a in p.items():
            for (p_, q_), b in r.items():
                cle = (i + p_, j + q_)
                sortie[cle] = sortie.get(cle, 0) + a * b * Fraction(1, q) ** (j * p_)
        # Les coefficients qui s'annulent apres somme ne sont pas gardes.
        return {cle: v for cle, v in sortie.items() if v}
    return mul


for q in (2, 3):
    mul = plan_quantique_mul(q)
    x = {(1, 0): Fraction(1)}
    y = {(0, 1): Fraction(1)}
    # 1. Controle de la relation definissante : xy - q*yx doit etre nul.
    gauche = mul(x, y)
    droite = {cle: q * v for cle, v in mul(y, x).items()}
    reste = {cle: gauche.get(cle, 0) - droite.get(cle, 0)
             for cle in set(gauche) | set(droite)}
    print(f"q = {q} : xy - q*yx =", {cle: v for cle, v in reste.items() if v})

    # 2. Rang des mots de longueur <= n, lus sur la base des x^i y^j de poids <= n,
    #    compare a la formule commutative (meme construction que pour A_1).
    for n in range(1, 6):
        base = [(i, j) for i in range(n + 1) for j in range(n + 1) if i + j <= n]
        index = {monome: k for k, monome in enumerate(base)}
        matrice = []
        for mot in mots_jusqua(n):
            element = {(0, 0): Fraction(1)}
            for lettre in mot:
                element = mul(element, x if lettre == "x" else y)
            ligne = [Fraction(0)] * len(base)
            for (i, j), v in element.items():
                ligne[index[(i, j)]] += v
            matrice.append(ligne)
        mesure = rang_sur_q(matrice)
        formule = (n + 1) * (n + 2) // 2
        print(f"  q = {q}, n = {n} : rang = {mesure}, formule = {formule} -> "
              f"{'identique' if mesure == formule else 'ECART'}")

q = 2 : xy - q*yx = {}
  q = 2, n = 1 : rang = 3, formule = 3 -> identique
  q = 2, n = 2 : rang = 6, formule = 6 -> identique
  q = 2, n = 3 : rang = 10, formule = 10 -> identique
  q = 2, n = 4 : rang = 15, formule = 15 -> identique
  q = 2, n = 5 : rang = 21, formule = 21 -> identique
q = 3 : xy - q*yx = {}
  q = 3, n = 1 : rang = 3, formule = 3 -> identique
  q = 3, n = 2 : rang = 6, formule = 6 -> identique
  q = 3, n = 3 : rang = 10, formule = 10 -> identique
  q = 3, n = 4 : rang = 15, formule = 15 -> identique
  q = 3, n = 5 : rang = 21, formule = 21 -> identique


Pour $q = 2$ comme pour $q = 3$, la relation est vérifiée à la machine (le dictionnaire des écarts est vide) et le rang suit la même formule. Changer $q$ ne change pas la dimension : le plan quantique est une **déformation plate** du plan commutatif du point de vue de la croissance.

C'est le premier exemple du carnet qui réunit les hypothèses du théorème et s'assoit exactement sur sa frontière basse. Le théorème interdit ]2, 3[ ; il n'interdit pas 2, et ce carnet exhibe un domaine gradué non commutatif qui l'atteint.

## 6. Le théorème du trou

Pour les algèbres générales, la dimension GK peut prendre n'importe quelle valeur réelle supérieure ou égale à 2 : les valeurs fractionnaires existent. Les trous apparaissent dès qu'on ajoute des hypothèses de structure.

| Résultat | Hypothèses | Conclusion |
|---|---|---|
| Bergman (dans Krause–Lenagan, Thm 2.5) | algèbre de type fini | aucune dimension GK dans $]1, 2[$ |
| Artin–Stafford, 1995 | domaines gradués par $\mathbb{N}$ | aucune dimension GK dans $]2, 11/5[$ |
| Smoktunowicz, 2006 | domaines gradués, composante de degré 0 de dimension finie | dimension GK $\le 2$ ou $\ge 3$ |

> **Théorème (Smoktunowicz, _Invent. Math._ 164 (2006), 635–640).** Une algèbre de type fini sur un corps, graduée par $\mathbb{N}$, sans diviseurs de zéro, dont la composante de degré zéro est de dimension finie, a pour dimension de Gelfand–Kirillov soit au plus 2, soit au moins 3.

La preuve publiée est une contradiction : supposer la dimension dans $]2, 3[$, en déduire que la composante graduée croît plus vite que linéairement et moins vite que quadratiquement, puis montrer que cette croissance sous-quadratique force en fait une croissance linéaire. La preuve publiée mobilise trois résultats profonds absents de Mathlib : le théorème du trou de Bergman, le théorème de Small–Warfield, et un argument de Riemann–Roch.

Le tableau suivant rassemble les familles calculées dans ce carnet et les confronte aux hypothèses.

In [10]:
# Une ligne par famille : ce que les cellules precedentes ont calcule (dim A_n,
# croissance, dimension de Gelfand-Kirillov), puis la branche du theoreme du trou
# ou cette famille se place, avec les hypotheses qu'elle satisfait ou non.
familles = [
    ("k[x,y]",                  "C(n+2,2)",       "polynomiale, degre 2", 2,       "graduee, integre",        "<= 2"),
    ("k[x,y,z]",                "C(n+3,3)",       "polynomiale, degre 3", 3,       "graduee, integre",        ">= 3"),
    ("algebre libre <x,y>",     "2^(n+1)-1",      "exponentielle",        "infinie", "graduee, integre",      ">= 3"),
    ("Weyl A_1",                "(n+1)(n+2)/2",   "polynomiale, degre 2", 2,       "integre, NON graduee",    "non applicable"),
    ("plan quantique k_q",      "(n+1)(n+2)/2",   "polynomiale, degre 2", 2,       "graduee, integre",        "<= 2 (frontiere)"),
]

entetes = ("famille", "dim A_n", "croissance", "GK", "hypotheses", "branche")
largeurs = (22, 15, 22, 8, 23, 15)  # largeur minimale de chaque colonne, pour aligner
ligne = " | ".join(t.ljust(l) for t, l in zip(entetes, largeurs))
print(ligne)
print("-" * len(ligne))
for ligne_famille in familles:
    # chaque valeur est convertie en texte puis completee a droite jusqu'a la largeur
    print(" | ".join(str(t).ljust(l) for t, l in zip(ligne_famille, largeurs)))

famille                | dim A_n         | croissance             | GK       | hypotheses              | branche        
------------------------------------------------------------------------------------------------------------------------
k[x,y]                 | C(n+2,2)        | polynomiale, degre 2   | 2        | graduee, integre        | <= 2           
k[x,y,z]               | C(n+3,3)        | polynomiale, degre 3   | 3        | graduee, integre        | >= 3           
algebre libre <x,y>    | 2^(n+1)-1       | exponentielle          | infinie  | graduee, integre        | >= 3           
Weyl A_1               | (n+1)(n+2)/2    | polynomiale, degre 2   | 2        | integre, NON graduee    | non applicable 
plan quantique k_q     | (n+1)(n+2)/2    | polynomiale, degre 2   | 2        | graduee, integre        | <= 2 (frontiere)


Lecture du tableau. Les deux anneaux de polynômes tombent dans les deux branches permises, aux degrés 2 et 3. L'algèbre libre tombe dans « au moins 3 » par l'infini. L'algèbre de Weyl a la bonne dimension mais pas les bonnes hypothèses : le théorème ne s'y applique pas, et elle n'est donc pas un contre-exemple. Le plan quantique est le seul témoin non commutatif du carnet qui réunit les hypothèses **et** atteint la valeur 2 — le bord exact de la zone permise.

Aucune de ces familles ne tombe dans $]2, 3[$, ce qui est cohérent, mais ce carnet ne **démontre** pas le théorème : il en calcule des instances et en énonce l'hypothèse. La démonstration est ailleurs — publiée en 2006, formalisée en 2026.

## 7. La preuve formelle de 2026 : ce qu'elle est, ce qu'elle n'est pas

La note de J. Stavrianos (18 novembre 2026) décrit la première formalisation connue du théorème, produite par un agent de preuve automatique travaillant seul dans un bac à sable équipé de Lean et Mathlib : une preuve de l'ordre de deux mille lignes, obtenue en une heure et demie d'horloge, pour un coût de l'ordre de la vingtaine de dollars au prix public. Aucune main humaine n'est intervenue pendant le run.

Le protocole de vérification est celui d'une salle blanche : l'énoncé gradué est reconstruit depuis une copie de confiance de la tâche, il doit coïncider exactement avec le théorème 1 de l'article (composante de degré zéro de dimension finie comprise), la preuve ne peut utiliser que les axiomes standard de Lean (`propext`, `Quot.sound`, `Classical.choice`), et un noyau indépendant la rejoue. La relecture humaine confirme en outre : compilation sans `sorry` ni avertissement, aucune décision native, aucun axiome neuf, aucun code `unsafe`, aucune redéfinition des objets du banc.

Le point le plus important est énoncé sans détour par la note : c'est une **preuve formelle du théorème**, pas une formalisation de la preuve publiée ; les deux ne coïncident que sur les étapes d'ouverture. La première moitié suit la route connue (croissance des pièces graduées, condition d'Ore, corps de fractions de degré zéro, génération par une seule pièce finie — les lemmes 1.6 et théorème 1.15 d'Artin–Stafford). La seconde moitié ne figure dans aucun des trois articles de référence : elle définit une **capacité**, le défaut $\dim(SU) - \dim U$ minimisé sur les sous-espaces contenant $U$, montre qu'elle se comporte comme le bord d'un graphe, qu'un plateau de capacité stabilise un sous-corps de division, et en déduit une borne linéaire sur $\dim A_n$ qui contredit l'hypothèse.

La note rattache cette quantité à des parents connus : la différence de volume qui définit le degré de transcendance inférieur de Zhang (1998), proposée par Zhang comme route vers ce même trou, et la méthode isopérimétrique de Hamidoune en combinatoire additive, adaptée aux sous-espaces de corps de division par Lecouvey (2014). Elle signale aussi un détail de conduite : l'agent a d'abord soumis une preuve partielle — cas commutatif complet, cas général troué par un `sorry` —, s'est vu refuser, a déposé un rapport d'échec honnête, puis a trouvé la route par la capacité dans les minutes suivantes.

Rien de tout cela ne se reproduit dans ce carnet. Il n'y a pas de Lean ici : le calcul des fonctions de Hilbert est un calcul, pas une preuve.

## 8. Ce que ce carnet calcule, ce qu'il ne calcule pas

**Calculé, avec sorties committées** : la fonction de Hilbert de $k[x,y]$, $k[x,y,z]$, de l'algèbre libre, de l'algèbre de Weyl et du plan quantique ; les pentes et les ratios qui discriminent croissance polynomiale et exponentielle ; le rang exact, sur $\mathbb{Q}$, de l'espace engendré par les mots de longueur au plus $n$ pour $A_1$ et pour le plan quantique, jusqu'à $n = 6$ et $n = 5$ respectivement.

**Non calculé, et assumé comme tel** : le théorème du trou lui-même. Ce carnet énonce ses hypothèses, exhibe des instances de ses deux branches, et mesure la frontière. Il ne prouve aucune des trois inclusions historiques (Bergman, Artin–Stafford, Smoktunowicz), et il ne reproduit pas la preuve formelle de 2026.

**La frontière de Mathlib**, telle que mesurée dans le corps de l'issue [#27589](https://github.com/jsboige/CoursIA/issues/27589) : aucune occurrence de la dimension de Gelfand–Kirillov, aucune structure d'algèbre graduée au-delà de `GradedMonoid`, aucune localisation d'Ore non commutative. C'est cette absence qui explique pourquoi la formalisation de 2026 a dû re-prouver elle-même la condition d'Ore et construire son corps de fractions, au lieu de les importer.

Le compagnon Lean de ce carnet (lemmes de croissance jouables, `lake build` vert, `sorry` comptés) est un livrable séparé du même grain : il ne vit pas dans ce carnet Python.

### Le compagnon Lean, nommé et montré

Le paragraphe précédent annonce le compagnon Lean ; cette sous-section le désigne. Il vit dans le dossier de la série, `formal_groups_lean/`, au module `FormalGroups/Growth.lean` : les lemmes de comptage sur lesquels reposent les calculs de ce carnet, prouvés en Lean 4 sur Mathlib, `lake build` vert.

| Section du carnet | Ce que le carnet calcule | Le même résultat, prouvé dans `Growth.lean` |
|---|---|---|
| §2 Anneaux de polynômes | $C(n+d, d)$ par énumération | `hilbertCumule d n` = `C(n + d, d)` ; `hilbertCumule_two_indet` (la croissance quadratique mesurée ici) ; `hilbertCumule_monotone` et `hilbertCumule_lt_succ` (la croissance est stricte) |
| §2 Couches de la filtration | — | `hilbertCouche d n` = `C(n + d − 1, d − 1)` ; `hilbertCumule_succ` (l'identité de Pascal, couche par couche) |
| §3 Algèbre libre | $1 + 2 + \dots + 2^n$ | `motsLibres d n` ; `motsLibres_deux` ($2^{n+1} - 1$, la valeur mesurée ici) ; `motsLibres_deux_ge_pow` (aucune borne polynomiale) |
| §2/§3 Le contraste | la pente qui décroît vers 0 | `hilbertCumule_deux_lt_motsLibres_deux` : dès le rang 2, la croissance polynomiale de $k[x, y]$ passe strictement sous la croissance exponentielle de $k\langle x, y\rangle$ |
| §1 La définition de la GK | — | `sous_multiplicative_le_pow` : sous-multiplicativité des dimensions, `dim A_n ≤ (dim A_1)^n` |

La frontière est la même des deux côtés, et elle est assumée : `Growth.lean` **ne définit pas** la dimension de Gelfand–Kirillov (Mathlib ne la porte pas — c'est la frontière mesurée plus haut), et il **ne prouve pas** le théorème du trou de Smoktunowicz. Il tient les faits de comptage que ce carnet rejoue numériquement, à la frontière exacte où le carnet s'arrête.

Rejouer la preuve formelle : depuis le dossier de la série, `lake build` dans `formal_groups_lean/` (toolchain et Mathlib de la série, jonction partagée).

## 9. Exercices

Trois exercices ferment le carnet. Les cellules sont des squelettes : elles s'exécutent sans erreur et n'imposent aucune solution. Les indices sont dans le commentaire de chaque cellule.

### Exercice 1 — La dimension de $k[x,y,z,w]$

**Objectif** : établir la dimension de Gelfand–Kirillov de l'anneau de polynômes à quatre variables, puis dire dans quelle branche du théorème il tombe.

**Étapes** : (1) calculer $\dim A_n$ par `monomes(4, n)` pour $n = 1$ à $8$ et comparer à $\binom{n+4}{4}$ ; (2) afficher les pentes avec `afficher(...)` et lire la limite ; (3) conclure : dimension GK, croissance, branche du théorème, hypothèses réunies ou non.

**Indice** : la machinerie de la section 2 se réutilise sans modification ; seul $d$ change.

In [11]:
# Exercice 1 -- TODO etudiant
# Etape 1 : dimensions = [monomes(4, n) for n in range(1, 9)]
# Etape 2 : comparer a [comb(n + 4, 4) for n in range(1, 9)]
# Etape 3 : afficher(...) puis conclure la dimension GK et la branche du theoreme
dimensions = None  # TODO etudiant
print("Exercice 1 a completer : dimension de Gelfand-Kirillov de k[x,y,z,w]")

Exercice 1 a completer : dimension de Gelfand-Kirillov de k[x,y,z,w]


### Exercice 2 — Le plan quantique à $q = 3$, et la platitude

**Objectif** : vérifier que changer $q$ ne change pas la dimension graduée, et expliquer pourquoi.

**Étapes** : (1) reprendre `plan_quantique_mul(3)` et vérifier la relation $xy - 3yx = 0$ ; (2) calculer le rang pour $n = 1$ à $6$ et comparer à $\frac{(n+1)(n+2)}{2}$ ; (3) expliquer en une phrase pourquoi la dimension ne dépend pas de $q$ (indice : la forme normale $x^i y^j$ et le compte des poids ne font intervenir que l'ordre des lettres, jamais la valeur de $q$ — seule l'indépendance linéaire pourrait en dépendre, et elle ne le fait pas pour $q \neq 0$).

**Indice** : la cellule de la section 5 contient déjà la boucle ; il suffit d'étendre la plage et de changer la valeur de $q$.

In [12]:
# Exercice 2 -- TODO etudiant
# Etape 1 : mul = plan_quantique_mul(3) puis verifier xy - 3*yx
# Etape 2 : rang jusqu'a n = 6, comparer a (n+1)(n+2)//2
# Etape 3 : rediger la phrase d'explication
resultat = None  # TODO etudiant
print("Exercice 2 a completer : platitude de la dimension en q pour le plan quantique")

Exercice 2 a completer : platitude de la dimension en q pour le plan quantique


### Exercice 3 — Deux algèbres monomiales, deux régimes

**Objectif** : classer deux algèbres définies par des mots interdits, et dire si le théorème du trou s'y applique.

Soit $A = k\langle x,y\rangle / (yx)$, où tout mot contenant la sous-chaîne `yx` est nul, et $B = k\langle x,y\rangle / (y^2)$. Dans $A$, les mots non nuls sont les $x^a y^b$, donc $\dim A_n = \frac{(n+1)(n+2)}{2}$ et la croissance est polynomiale. Dans $B$, les mots non nuls sont ceux sans `yy` consécutif, dont le nombre de longueur exactement $n$ est le nombre de Fibonacci $F_{n+2}$ — croissance exponentielle.

**Étapes** : (1) compter par énumération les mots de longueur au plus $n$ sans la sous-chaîne interdite, pour les deux algèbres et $n = 1$ à $10$ ; (2) afficher pentes et ratios ; (3) conclure la dimension GK de chacune ; (4) vérifier que ni l'une ni l'autre n'est un domaine (exhiber le produit nul) et expliquer pourquoi le théorème du trou ne les contraint pas.

**Indice** : filtrer `mots_jusqua(n)` avec `"yx" not in mot` et `"yy" not in mot` ; pour le point (4), `yx = 0` dans $A$ et `y * y = 0` dans $B$ sont des produits de deux éléments non nuls.

In [13]:
# Exercice 3 -- TODO etudiant
# Etape 1 : compter les mots sans "yx" et sans "yy" pour n = 1..10
# Etape 2 : afficher(...) les deux suites
# Etape 3 : conclure les dimensions GK ; comparer a la suite de Fibonacci pour B
# Etape 4 : exhiber un produit nul dans chaque algebre et conclure sur les hypotheses
resultat = None  # TODO etudiant
print("Exercice 3 a completer : deux algebres monomiales, deux regimes de croissance")

Exercice 3 a completer : deux algebres monomiales, deux regimes de croissance


## Conclusion

Le carnet a calculé ce qui se calcule : les fonctions de Hilbert de cinq familles d'algèbres, leurs pentes de croissance, et des rangs exacts qui donnent la dimension de l'espace engendré par les mots jusqu'à un degré borné. Il a énoncé le théorème du trou avec ses trois étages historiques, et situé chaque famille par rapport à ses hypothèses : deux anneaux de polynômes dans les deux branches permises, l'algèbre libre à l'infini, l'algèbre de Weyl hors hypothèses, le plan quantique sur la frontière basse.

Il n'a pas prouvé le théorème, et il le dit. La preuve publiée de 2006 et sa formalisation de 2026 restent extérieures au dépôt ; la seconde est désormais décrite publiquement, avec son statut exact — preuve formelle du théorème, pas transcription formelle de la preuve publiée.

La suite naturelle est le compagnon Lean : les lemmes de croissance jouables (filtration par la longueur des mots, monotonie, comparaison degré/longueur) que Mathlib ne fournit pas encore.

## Références

- M. Artin, J. T. Stafford, *Noncommutative graded domains with quadratic growth*, Inventiones Mathematicae 122 (1995), 231–276.
- G. R. Krause, T. H. Lenagan, *Growth of Algebras and Gelfand–Kirillov Dimension*, Graduate Studies in Mathematics 22, American Mathematical Society, 2000 (réédition).
- C. Lecouvey, *Plünnecke and Kneser type theorems for dimension estimates*, Combinatorica (2014), arXiv:1109.0134.
- L. W. Small, R. B. Warfield Jr., *Prime affine algebras of Gelfand–Kirillov dimension one*, Journal of Algebra 91 (1984), 386–389.
- A. Smoktunowicz, *The Artin–Stafford gap theorem*, Proceedings of the American Mathematical Society 133 (2005), 1925–1928.
- A. Smoktunowicz, *There are no graded domains with GK dimension strictly between 2 and 3*, Inventiones Mathematicae 164 (2006), 635–640.
- J. Stavrianos, note de vérification sur la formalisation du théorème de Smoktunowicz, 18 novembre 2026 — note publique, non évaluée par les pairs ; elle est la source du récit de la section 7.
- J. J. Zhang, *On lower transcendence degree*, Advances in Mathematics 139 (1998), 157–193.

Grain : point 1 de [#27589](https://github.com/jsboige/CoursIA/issues/27589). Îlot : [GROTH](README.md).